In [ ]:
# =============================================================
# EPN — EVOKED THETA TEST   [new RAIN_IRC data + FIR-175 GT, matched to BOB]
# =============================================================
# Reads the SAME npz splits BOB reads: RAIN_IRC/data splits/onset (native 300 Hz),
# channel = best_ch_idx from the npz, cal/test = the npz split array, GT = 4-8 Hz
# zero-phase FIR-175 (byte-identical taps to BOB). The EPN model (Liu et al. 2025)
# architecture is unchanged; the sample-denominated windowing is scaled to 300 Hz
# (INPUT_MS stays 500 ms -> 150 samples; pool/step preserve their 600 Hz durations).
# Output: one model-tagged JSON in the BOB schema (raw per-timepoint errors in radians).

import numpy as np
import os, time, json, gc, glob, warnings
from scipy.signal import butter, hilbert, filtfilt, lfilter, lfilter_zi, firwin, welch
from scipy.stats import wilcoxon, ttest_rel, pearsonr, spearmanr

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch device: {DEVICE}")

# ---- DATA + GT (same as BOB) ----
STIM_DIR = '/content/drive/MyDrive/RAIN_IRC/data splits/onset'

HORIZON_MS = 0.0           # <-- 0.0 or 50.0
POWER_GATE = False         # <-- True = gated, False = ungated
POWER_GATE_PERCENTILE = 0  # <-- only used when POWER_GATE = True
gate_tag = 'gated' if POWER_GATE else 'ungated'
OUTDIR = f'/content/drive/MyDrive/RAIN_IRC/results/EPN_onset'
os.makedirs(OUTDIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(STIM_DIR, '*.npz')))
assert files, f"no npz found in {STIM_DIR}"
_d0 = np.load(files[0], allow_pickle=True)
FS = float(_d0['fs']); PRE_SAMP = int(_d0['onset_in_epoch']); _d0.close()

EVAL_TIMES_S    = [0.3, 0.4, 0.5, 0.6, 0.7]
EVAL_TIMES_SAMP = [PRE_SAMP + int(t * FS) for t in EVAL_TIMES_S]
HORIZON_SAMP    = int(HORIZON_MS * FS / 1000.0)

# --- Ground truth: 4-8 Hz zero-phase FIR-175 (filtfilt), identical taps to BOB ---
GT_LOW, GT_HIGH = 4.0, 8.0
GT_FIR_NUMTAPS  = 175                            # 300 Hz (half of 351 @ 600 Hz)
GT_HALF_S       = (GT_FIR_NUMTAPS // 2) / FS     # ~0.29 s acausal reach
THETA_SEARCH_LO, THETA_SEARCH_HI = 4.0, 8.0

# EPN config (Liu et al. 2025). Architecture unchanged; sample params scaled to 300 Hz.
INPUT_MS        = 500
INPUT_SAMPLES   = int(INPUT_MS * FS / 1000.0)    # 150 @ 300 Hz (was 300 @ 600 Hz)
AVG_POOL_KERNEL = 13                             # 25 @ 600 Hz -> 13 @ 300 Hz (same ~42 ms)
HIDDEN_DIM      = 500
DROPOUT_P       = 0.9
N_EPOCHS        = 3000
EPN_LR          = 1e-5
BATCH_SIZE      = 512
TRAIN_STEP      = 3                              # 6 @ 600 Hz -> 3 @ 300 Hz (same 10 ms stride)

SEED = 42

print()
print("=" * 100)
print("EPN — EVOKED THETA TEST  (new RAIN_IRC 300 Hz data, FIR-175 GT, matched to BOB)")
print(f"  Horizon: {HORIZON_MS}ms | Power gate: "
      + (f"ON ({POWER_GATE_PERCENTILE}th pctl)" if POWER_GATE else "OFF"))
print(f"  FS={FS:.0f} Hz | t=0 at sample {PRE_SAMP} | input {INPUT_MS}ms={INPUT_SAMPLES} samp"
      f" | pool {AVG_POOL_KERNEL} | step {TRAIN_STEP}")
print(f"  GT: firwin({GT_FIR_NUMTAPS}, [{GT_LOW},{GT_HIGH}], hamming, fs={FS:.0f})"
      f" -> filtfilt -> Hilbert   (half-length {GT_HALF_S:.3f}s)")
print(f"  Data: {STIM_DIR}  ({len(files)} subjects)")
print(f"  Output: {OUTDIR}")
print("=" * 100)


# =============================================================
# SHARED FUNCTIONS
# =============================================================

def causal_envelope_fast(raw_1d, filt_low, filt_high):
    """Vectorized causal envelope (lfilter-based) for the power gate. [UNCHANGED]"""
    nyq = FS / 2.0
    b, a = butter(1, [max(0.5, filt_low)/nyq,
                       min(filt_high, nyq-0.5)/nyq], btype='band')
    zi = lfilter_zi(b, a) * raw_1d[0]
    filt, _ = lfilter(b, a, raw_1d, zi=zi)
    alpha = 2.0 / (int(0.1 * FS) + 1)
    sq = filt.astype(np.float64) ** 2
    b_e, a_e = np.array([alpha]), np.array([1.0, -(1.0 - alpha)])
    zi_e = lfilter_zi(b_e, a_e) * sq[0]
    env, _ = lfilter(b_e, a_e, sq, zi=zi_e)
    return np.sqrt(np.maximum(env, 0)).astype(np.float32)

def compute_matched_gt_batch(raw_1ch, trial_indices, filt_low, filt_high):
    """Evoked-theta GT: linear-phase FIR, zero-phase (filtfilt).
    firwin(175,[4,8],hamming,fs=300) -> byte-identical taps to BOB."""
    gt = {}
    nyq = FS / 2.0
    low  = max(0.5, min(filt_low, nyq - 1))
    high = max(low + 0.5, min(filt_high, nyq - 0.5))
    taps = firwin(GT_FIR_NUMTAPS, [low, high], pass_zero=False,
                  fs=FS, window='hamming')
    for tr in trial_indices:
        f = filtfilt(taps, [1.0], raw_1ch[tr].astype(np.float64))   # zero-phase
        ph = np.angle(hilbert(f))
        gt[tr] = (np.sin(ph).astype(np.float32),
                  np.cos(ph).astype(np.float32))
    return gt

# ---- metrics: BOB-identical (MACE, PLV, CSD, Acc%, bias) ----
def errors_to_metrics(errors):
    if len(errors) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    mean_abs = float(np.mean(np.abs(errors))); R = float(np.abs(np.mean(np.exp(1j * errors))))
    return {'mace': float(np.degrees(mean_abs)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),   # circular SD (deg)
            'acc': float((1.0 - mean_abs / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * errors))))),
            'n_samples': int(len(errors))}

def itpc_from_gt(gt, trials, esamp):
    """GT-phase ITPC across trials at one eval sample (BOB-identical basis)."""
    ph = []
    for tr in trials:
        if tr in gt:
            gs, gc = gt[tr]
            if esamp < len(gs): ph.append(np.arctan2(gs[esamp], gc[esamp]))
    if len(ph) < 2: return float('nan')
    return float(np.abs(np.mean(np.exp(1j * np.array(ph)))))

def estimate_theta_peak(raw_1ch, onset_samp):
    seg = raw_1ch[:, onset_samp:]
    if seg.shape[1] < 16: return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= THETA_SEARCH_LO) & (f <= THETA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan

Mounted at /content/drive
PyTorch device: cpu

EPN — EVOKED THETA TEST  (new RAIN_IRC 300 Hz data, FIR-175 GT, matched to BOB)
  Horizon: 0.0ms | Power gate: OFF
  FS=300 Hz | t=0 at sample 300 | input 500ms=150 samp | pool 13 | step 3
  GT: firwin(175, [4.0,8.0], hamming, fs=300) -> filtfilt -> Hilbert   (half-length 0.290s)
  Data: /content/drive/MyDrive/RAIN_IRC/data splits/onset  (20 subjects)
  Output: /content/drive/MyDrive/RAIN_IRC/results/EPN_onset


In [ ]:
# =============================================================
# EPN (Liu et al. 2025) — model + train/evaluate   [MODEL UNCHANGED]
# =============================================================

class EPN(nn.Module):
    def __init__(self, n_in, pool_k=13, hidden=500, drop=0.9):
        super().__init__()
        self.n_in = n_in
        self.pool = nn.AvgPool1d(pool_k, stride=1, padding=pool_k // 2)
        self.fc1 = nn.Linear(n_in, hidden)
        self.fc2 = nn.Linear(hidden, 2)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        x = self.pool(x.unsqueeze(1)).squeeze(1)[:, :self.n_in]
        return self.fc2(self.drop(torch.relu(self.fc1(x))))

def build_epn_data(raw_1ch, gt, envs, gate_thresh, trials, epoch_len, dense=False):
    X, Y, M = [], [], []
    # GT half-length ~0.29 s -> targets before ~0.3 s are onset-smeared. Start dense
    # training at the eval floor (0.3 s). Range/step are in time (scale with FS).
    tps = (list(range(PRE_SAMP + int(0.30*FS), PRE_SAMP + int(0.85*FS), TRAIN_STEP))
           if dense else EVAL_TIMES_SAMP)
    for tr in trials:
        if tr not in gt: continue
        gs, gc = gt[tr]
        env = envs.get(tr)
        for es in tps:
            gt_s = es + HORIZON_SAMP
            raw_s = es - INPUT_SAMPLES
            if raw_s < 0 or gt_s >= epoch_len: continue
            if POWER_GATE and env is not None and env[es] < gate_thresh: continue
            bd = np.diff(raw_1ch[tr, raw_s:es+1].astype(np.float64)).astype(np.float32)
            X.append(bd); Y.append([gs[gt_s], gc[gt_s]])
            if not dense: M.append(EVAL_TIMES_S[EVAL_TIMES_SAMP.index(es)])
    if not X: return np.zeros((0, INPUT_SAMPLES), np.float32), np.zeros((0, 2), np.float32), M
    return np.array(X, np.float32), np.array(Y, np.float32), M

def circ_errors(ps, pc, gs, gc):
    return np.arctan2(ps*gc - pc*gs, pc*gc + ps*gs)


print(f"{'='*96}")
print("RUNNING EPN  (no bias correction)")
print(f"{'='*96}")
print(f"{'Subject':<14s} {'N':>4s} {'MACE':>7s} {'CSD':>7s} {'Acc%':>6s} {'PLV':>6s} {'Bias':>7s} {'SNR':>5s} {'ITPC':>6s} {'Peak':>6s}")
print("-" * 86)

epn_results = []
torch.manual_seed(SEED); np.random.seed(SEED)

for fp in files:
    t0 = time.time(); d = np.load(fp, allow_pickle=True); name = os.path.basename(fp)[:-4]
    if float(d['fs']) != FS or int(d['onset_in_epoch']) != PRE_SAMP:
        print(f"{name}: geometry mismatch, skipping"); d.close(); continue
    epochs = d['epochs']; best = int(d['best_ch_idx']); ch_name = str(d['best_ch_name'])
    split_arr = np.array([str(x) for x in d['split']]); theta_snr = float(d['ch_theta_snr'][best])
    raw_1ch = epochs[:, best, :].astype(np.float32)
    n_trials, n_ch, epoch_len = epochs.shape
    cal_trials = np.where(split_arr == 'cal')[0]
    test_trials = np.where(split_arr == 'test')[0]
    all_trials = np.arange(n_trials)

    # GT (FIR-175, same as BOB) + envelopes for the gate
    gt_all = compute_matched_gt_batch(raw_1ch, all_trials, GT_LOW, GT_HIGH)
    envs = {}; gv = []
    for tr in all_trials:
        envs[tr] = causal_envelope_fast(raw_1ch[tr], GT_LOW, GT_HIGH)
    for tr in cal_trials:
        for es in EVAL_TIMES_SAMP:
            if es < len(envs[tr]): gv.append(envs[tr][es])
    gate_thresh = np.percentile(gv, POWER_GATE_PERCENTILE) if gv else 0.0

    # per-subject theta descriptors (BOB-identical: GT-phase ITPC on TEST + Welch peak)
    itpc_tp = {t: (itpc_from_gt(gt_all, test_trials, es) if es < epoch_len else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values())))
    f_peak = estimate_theta_peak(raw_1ch, PRE_SAMP)

    # Training data
    X_tr, Y_tr, _ = build_epn_data(raw_1ch, gt_all, envs, gate_thresh,
                                     cal_trials, epoch_len, dense=True)
    n_train = len(X_tr)
    print(f"  {name}: {n_train} train samples", end='', flush=True)

    if n_train < 50:
        print(" | SKIP")
        epn_results.append({
            'name': name, 'best_ch': ch_name, 'n_trials': int(n_trials),
            'n_test_samples': 0, 'n_train_samples': 0, 'theta_snr': theta_snr,
            'theta_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
            'theta_itpc_mean': itpc_mean,
            'theta_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
            'metrics_pooled': errors_to_metrics(np.array([])),
            'metrics_per_timepoint': {str(t): errors_to_metrics(np.array([])) for t in EVAL_TIMES_S},
            'errors_per_timepoint_rad': {str(t): [] for t in EVAL_TIMES_S},
            'compute_time_ms': 0.0})
        d.close(); del epochs, raw_1ch, gt_all, envs; gc.collect(); continue

    Xm, Xs = X_tr.mean(0, keepdims=True), X_tr.std(0, keepdims=True) + 1e-8
    X_trn = (X_tr - Xm) / Xs

    model = EPN(INPUT_SAMPLES, AVG_POOL_KERNEL, HIDDEN_DIM, DROPOUT_P).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=EPN_LR)
    loss_fn = nn.MSELoss()
    Xt = torch.FloatTensor(X_trn).to(DEVICE)
    Yt = torch.FloatTensor(Y_tr).to(DEVICE)
    dl = DataLoader(TensorDataset(Xt, Yt), batch_size=BATCH_SIZE, shuffle=True)

    model.train()
    for ep in range(N_EPOCHS):
        for xb, yb in dl:
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
        if (ep+1) % 500 == 0: print(f" | ep{ep+1}", end='', flush=True)

    # Evaluate test (NO bias correction -- raw test errors)
    X_te, Y_te, te_m = build_epn_data(raw_1ch, gt_all, envs, gate_thresh,
                                        test_trials, epoch_len, dense=False)
    if len(X_te) > 0:
        X_ten = (X_te - Xm) / Xs
        model.eval()
        with torch.no_grad():
            pred = model(torch.FloatTensor(X_ten).to(DEVICE)).cpu().numpy()
        test_errs = circ_errors(pred[:,0], pred[:,1], Y_te[:,0], Y_te[:,1])
    else:
        test_errs = np.array([])

    met = errors_to_metrics(test_errs)

    per_tp = {t: [] for t in EVAL_TIMES_S}
    for i, es in enumerate(te_m): per_tp[es].append(test_errs[i])
    per_tp = {t: np.array(e) for t, e in per_tp.items()}
    per_tp_met = {t: errors_to_metrics(e) for t, e in per_tp.items()}

    # Timing (per-sample inference latency)
    tsamps = []
    if len(X_te) > 0:
        ti = torch.FloatTensor(X_ten[:1]).to(DEVICE)
        model.eval()
        with torch.no_grad():
            for _ in range(200):
                ts = time.perf_counter(); model(ti)
                if DEVICE.type == 'cuda': torch.cuda.synchronize()
                tsamps.append((time.perf_counter()-ts)*1000)

    dt = time.time() - t0
    pk = f"{f_peak:.2f}" if np.isfinite(f_peak) else " n/a"
    print(f"\r{name:<14s} {met['n_samples']:>4d} {met['mace']:>6.1f}° {met['csd']:>6.1f}° "
          f"{met['acc']:>5.1f}% {met['plv']:>5.3f} {met['bias']:>+6.1f}° "
          f"{theta_snr:>5.2f} {itpc_mean:>6.3f} {pk:>6s}  [{dt:.0f}s]")

    epn_results.append({
        'name': name, 'best_ch': ch_name, 'n_trials': int(n_trials),
        'n_test_samples': met['n_samples'], 'n_train_samples': int(n_train),
        'theta_snr': theta_snr,
        'theta_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'theta_itpc_mean': itpc_mean,
        'theta_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'metrics_pooled': met,
        'metrics_per_timepoint': {str(t): per_tp_met[t] for t in EVAL_TIMES_S},
        'errors_per_timepoint_rad': {str(t): [float(x) for x in per_tp[t]] for t in EVAL_TIMES_S},
        'compute_time_ms': float(np.mean(tsamps)) if tsamps else 0.0})

    d.close(); del epochs, raw_1ch, gt_all, envs, model, Xt, Yt
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    gc.collect()

print(f"\n{len(epn_results)} subjects done")

RUNNING EPN  (no bias correction)
Subject           N    MACE     CSD   Acc%    PLV    Bias   SNR   ITPC   Peak
--------------------------------------------------------------------------------------
Subject10001    250   49.8°   64.2°  72.3% 0.533   +0.0°  2.56  0.097   4.03  [488s]
Subject1001     250   46.6°   59.5°  74.1% 0.583   -7.8°  2.59  0.162   4.03  [460s]
Subject11002    250   48.8°   61.7°  72.9% 0.560   -3.6°  2.04  0.128   4.03  [455s]
Subject12001    250   46.7°   60.3°  74.1% 0.575   -6.1°  1.93  0.179   4.03  [454s]
Subject13001    250   40.2°   52.2°  77.7% 0.660   +3.9°  2.65  0.072   6.88  [454s]
Subject14001    250   50.9°   65.5°  71.7% 0.520   -5.4°  2.41  0.169   4.03  [476s]
Subject15001    250   54.7°   68.7°  69.6% 0.487  -12.2°  2.36  0.146   4.03  [453s]
Subject16001    250   49.2°   63.2°  72.7% 0.544   -0.7°  1.74  0.192   4.03  [450s]
Subject17001    250   47.0°   60.1°  73.9% 0.576   -3.9°  2.45  0.205   4.03  [455s]
Subject18002    250   50.2°   64.2° 

In [ ]:
# =============================================================
# SUMMARY + SAVE  (complete model-tagged JSON, BOB-parallel)
# =============================================================
valid = [r for r in epn_results if r['n_test_samples'] >= 10]
n = len(valid)
M  = np.array([r['metrics_pooled']['mace'] for r in valid])
P  = np.array([r['metrics_pooled']['plv'] for r in valid])
Cd = np.array([r['metrics_pooled']['csd'] for r in valid])
A  = np.array([r['metrics_pooled']['acc'] for r in valid])
SNR = np.array([r['theta_snr'] for r in valid])
IT  = np.array([r['theta_itpc_mean'] for r in valid])

print('=' * 72); print('EPN THETA — group summary'); print('=' * 72)
print(f"  n={n} participants (>=10 test samples)")
print(f"  MACE {M.mean():.1f}°±{M.std():.1f}   PLV {P.mean():.3f}   CSD {Cd.mean():.1f}°   Acc {A.mean():.1f}%")

print('\n--- Per-timepoint ---')
print(f"{'ms':>5}{'MACE':>9}{'PLV':>8}{'CSD':>8}{'Acc':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    g = lambda k: np.array([r['metrics_per_timepoint'][str(t)][k] for r in valid])
    ti = np.array([r['theta_itpc_per_timepoint'][str(t)] for r in valid])
    print(f"{int(t*1000):>5}{g('mace').mean():>8.1f}°{g('plv').mean():>8.3f}"
          f"{g('csd').mean():>7.1f}°{g('acc').mean():>7.1f}%{np.nanmean(ti):>8.3f}{g('n_samples').mean():>7.0f}")

group_stats = {'n_participants': int(n),
               'mean_mace': float(M.mean()), 'median_mace': float(np.median(M)),
               'mean_plv': float(P.mean()), 'mean_csd': float(Cd.mean()), 'mean_acc': float(A.mean())}

print('\n--- Accuracy correlations across subjects ---')
if n >= 3 and np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, A); rs1, ps1 = spearmanr(SNR, A)
    group_stats.update({'acc_vs_snr_pearson_r': float(r1), 'acc_vs_snr_pearson_p': float(p1),
                        'acc_vs_snr_spearman_r': float(rs1), 'acc_vs_snr_spearman_p': float(ps1)})
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if n >= 3 and np.std(IT) > 0:
    r2, p2 = pearsonr(IT, A); rs2, ps2 = spearmanr(IT, A)
    group_stats.update({'acc_vs_itpc_pearson_r': float(r2), 'acc_vs_itpc_pearson_p': float(p2),
                        'acc_vs_itpc_spearman_r': float(rs2), 'acc_vs_itpc_spearman_p': float(ps2)})
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

out = {'model': 'EPN',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': HORIZON_MS,
                  'gt_band': [GT_LOW, GT_HIGH], 'gt_fir_numtaps': GT_FIR_NUMTAPS,
                  'input_ms': INPUT_MS, 'input_samples': INPUT_SAMPLES, 'avg_pool_kernel': AVG_POOL_KERNEL,
                  'train_step': TRAIN_STEP, 'n_epochs': N_EPOCHS, 'lr': EPN_LR,
                  'hidden_dim': HIDDEN_DIM, 'dropout_p': DROPOUT_P, 'power_gate': POWER_GATE,
                  'power_gate_percentile': POWER_GATE_PERCENTILE if POWER_GATE else None,
                  'data_dir': STIM_DIR, 'split': 'chronological cal/test from npz split array',
                  'channel': 'best_ch_idx from npz', 'conditions': ['model'],
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians',
                  'note': ('single-condition model; headline metrics under metrics_pooled / '
                           'metrics_per_timepoint / errors_per_timepoint_rad (analogue of BOB opt_*)')},
       'group_stats': group_stats,
       'subjects': epn_results}
outpath = os.path.join(OUTDIR, 'epn_theta_results.json')
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f"\nSaved -> {outpath}")
print("Per subject: metrics_pooled + per-timepoint metrics, per-timepoint errors (rad), SNR, ITPC, peak.")
print("Same data/GT/channel/split as BOB -> join on 'name'; EPN.metrics_pooled <-> BOB.opt_metrics_pooled.")

EPN THETA — group summary
  n=20 participants (>=10 test samples)
  MACE 48.2°±3.8   PLV 0.561   CSD 61.6°   Acc 73.2%

--- Per-timepoint ---
   ms     MACE     PLV     CSD     Acc    ITPC      N
  300    48.6°   0.560   61.8°   73.0%   0.121     50
  400    49.9°   0.547   63.1°   72.3%   0.130     50
  500    49.3°   0.553   62.4°   72.6%   0.157     50
  600    47.3°   0.574   60.5°   73.7%   0.157     50
  700    46.0°   0.594   58.5°   74.5%   0.138     50

--- Accuracy correlations across subjects ---
  Acc vs SNR : r=+0.060 (p=0.8007)  rho=+0.182 (p=0.4426)
  Acc vs ITPC: r=+0.294 (p=0.2086)  rho=+0.229 (p=0.3324)

Saved -> /content/drive/MyDrive/RAIN_IRC/results/EPN_onset/epn_theta_results.json
Per subject: metrics_pooled + per-timepoint metrics, per-timepoint errors (rad), SNR, ITPC, peak.
Same data/GT/channel/split as BOB -> join on 'name'; EPN.metrics_pooled <-> BOB.opt_metrics_pooled.
